# Ranking metrics (NDCG, MAP, MRR) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def cosine(a,b):
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float)
    return float(a@b/(np.linalg.norm(a)*np.linalg.norm(b)))
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x)))
def dcg(rels): return sum((2**r-1)/math.log2(i+2) for i,r in enumerate(rels))
def ndcg(rels):
    best=sorted(rels, reverse=True)
    return dcg(rels)/dcg(best) if dcg(best)>0 else 0.0
print('setup ok')

## Measuring ranked lists by position-aware usefulness
Ranking metrics reward putting useful items early. The examples compute DCG/NDCG, show the discount curve, compute average precision, compute reciprocal rank, and compare two slates with the same hits in different positions.

In [ ]:
# 1) DCG and NDCG for graded relevance
rels=[3,2,0,1]; D=dcg(rels); N=ndcg(rels)
plt.figure(figsize=(6,3)); plt.bar(['DCG','ideal DCG'],[D,dcg(sorted(rels,reverse=True))]); plt.title(f'NDCG = {N:.3f}')
assert abs(D-9.323465818787767)<1e-9 and abs(N-0.992619504174702)<1e-9

In [ ]:
# 2) logarithmic discounts make early positions matter most
disc=np.array([1/math.log2(i+2) for i in range(10)])
plt.figure(figsize=(6,3)); plt.plot(np.arange(1,11),disc,'-o'); plt.xlabel('rank'); plt.ylabel('discount'); plt.title('NDCG discount curve')
assert abs(disc[0]-1.0)<1e-12 and disc[4]<0.4

In [ ]:
# 3) average precision averages precision at hit positions
hits=np.array([1,0,1,1,0]); prec=np.cumsum(hits)/(np.arange(len(hits))+1); AP=float((prec*hits).sum()/hits.sum())
plt.figure(figsize=(6,3)); plt.step(np.arange(1,6),prec,where='post'); plt.scatter(np.where(hits)[0]+1,prec[hits==1],c='r'); plt.title(f'AP = {AP:.3f}')
assert abs(AP-0.8055555555555555)<1e-12

In [ ]:
# 4) MRR cares only about the first relevant result
hits=np.array([0,0,1,1,0]); rr=1/(np.argmax(hits)+1) if hits.any() else 0
plt.figure(figsize=(6,3)); plt.bar(np.arange(1,6),hits); plt.title(f'first hit at rank 3 -> MRR {rr:.3f}')
assert abs(rr-1/3)<1e-12

In [ ]:
# 5) same number of hits, different order: metrics reward early hits
A=[1,1,0,0,0]; B=[0,0,0,1,1]
def ap(h):
    h=np.array(h); p=np.cumsum(h)/(np.arange(len(h))+1); return float((p*h).sum()/h.sum())
vals=[ap(A),ap(B),ndcg(A),ndcg(B)]
plt.figure(figsize=(6,3)); plt.bar(['AP early','AP late','NDCG early','NDCG late'],vals); plt.xticks(rotation=20); plt.title('order matters')
assert vals[0]>vals[1] and vals[2]>vals[3]